# Building a Stock Analysis MCP Server from Scratch

This notebook is the complete companion to the To Data & Beyond tutorial. It builds a local Model Context Protocol (MCP) server that retrieves market data through `yfinance` and exposes three tools plus one resource to MCP clients such as Claude Desktop.

> Educational use only. Market data may be delayed or incomplete, and nothing in this notebook is financial advice.

## What you will build

The server exposes:

- `get_stock_price`: latest available closing price for one ticker
- `get_stock_history`: historical prices returned as CSV
- `compare_stocks`: side-by-side comparison of two latest closing prices
- `stock://{symbol}`: a readable MCP resource for one ticker

The flow is `Claude Desktop → MCP server → yfinance → structured result`.

## SDK version note

The original May 2025 article used MCP Python SDK v1 and imported `FastMCP` from `mcp.server.fastmcp`. The current stable v2 SDK renamed that class to `MCPServer`. This notebook uses the current v2 API. If you need to reproduce the original code exactly, install `mcp>=1.28,<2` instead.

In [ ]:
%pip install -q "mcp[cli]>=2,<3" yfinance

## Create the MCP server

Run the next cell once. It writes a standalone `stock_price_server.py` file in the notebook's working directory. Input validation keeps ticker symbols and history periods bounded, while concise exceptions return useful MCP tool errors.

In [ ]:
%%writefile stock_price_server.py
from __future__ import annotations

from dataclasses import dataclass

import yfinance as yf
from mcp.server import MCPServer


mcp = MCPServer(
    "stock-analysis",
    title="Stock Analysis MCP Server",
    description="Read recent public market data for educational analysis.",
    instructions=(
        "Use these read-only tools to inspect recent stock prices and history. "
        "Always mention that market data may be delayed and is not financial advice."
    ),
    version="1.0.0",
)

ALLOWED_PERIODS = {"1d", "5d", "1mo", "3mo", "6mo", "1y", "2y", "5y"}


@dataclass(frozen=True)
class LatestQuote:
    symbol: str
    price: float
    currency: str
    market_date: str


def _normalize_symbol(symbol: str) -> str:
    normalized = symbol.strip().upper()
    if not normalized or len(normalized) > 15:
        raise ValueError("Ticker symbol must contain between 1 and 15 characters.")
    if not all(character.isalnum() or character in {".", "-", "^"} for character in normalized):
        raise ValueError("Ticker symbol contains unsupported characters.")
    return normalized


def _latest_quote(symbol: str) -> LatestQuote:
    normalized = _normalize_symbol(symbol)
    ticker = yf.Ticker(normalized)
    history = ticker.history(period="5d", auto_adjust=False)
    closes = history["Close"].dropna() if "Close" in history else history
    if history.empty or closes.empty:
        raise ValueError(f"No recent price data was returned for {normalized}.")

    currency = "USD"
    try:
        currency = str(ticker.fast_info.get("currency") or currency)
    except Exception:
        pass

    return LatestQuote(
        symbol=normalized,
        price=round(float(closes.iloc[-1]), 2),
        currency=currency,
        market_date=str(closes.index[-1].date()),
    )


@mcp.tool()
def get_stock_price(symbol: str) -> dict[str, str | float]:
    """Return the latest available closing price for a ticker symbol."""
    quote = _latest_quote(symbol)
    return {
        "symbol": quote.symbol,
        "price": quote.price,
        "currency": quote.currency,
        "market_date": quote.market_date,
        "notice": "Market data may be delayed. Educational use only.",
    }


@mcp.resource("stock://{symbol}")
def stock_resource(symbol: str) -> str:
    """Expose a readable latest-price summary as an MCP resource."""
    quote = _latest_quote(symbol)
    return (
        f"{quote.symbol} closed at {quote.price:.2f} {quote.currency} "
        f"on {quote.market_date}. Market data may be delayed."
    )


@mcp.tool()
def get_stock_history(symbol: str, period: str = "1mo") -> str:
    """Return historical OHLCV data as CSV for an allowed yfinance period."""
    normalized = _normalize_symbol(symbol)
    normalized_period = period.strip().lower()
    if normalized_period not in ALLOWED_PERIODS:
        allowed = ", ".join(sorted(ALLOWED_PERIODS))
        raise ValueError(f"Unsupported period. Choose one of: {allowed}.")

    history = yf.Ticker(normalized).history(
        period=normalized_period,
        auto_adjust=False,
    )
    if history.empty:
        raise ValueError(f"No historical data was returned for {normalized}.")

    columns = [column for column in ["Open", "High", "Low", "Close", "Volume"] if column in history]
    return history[columns].tail(100).to_csv()


@mcp.tool()
def compare_stocks(symbol1: str, symbol2: str) -> dict[str, object]:
    """Compare the latest available closing prices of two ticker symbols."""
    first = _latest_quote(symbol1)
    second = _latest_quote(symbol2)
    if first.currency != second.currency:
        comparison = "Prices use different currencies and should not be compared directly."
        percent_difference = None
    else:
        leader, follower = (first, second) if first.price >= second.price else (second, first)
        percent_difference = round(((leader.price - follower.price) / follower.price) * 100, 2)
        comparison = f"{leader.symbol} is {percent_difference:.2f}% above {follower.symbol} by latest close."

    return {
        "first": first.__dict__,
        "second": second.__dict__,
        "percent_difference": percent_difference,
        "comparison": comparison,
        "notice": "Market data may be delayed. Educational use only.",
    }


if __name__ == "__main__":
    mcp.run()


## Optional local data smoke test

This checks the data helper without starting the MCP transport. It needs internet access and current Yahoo Finance data.

In [ ]:
from stock_price_server import _latest_quote

for ticker_symbol in ("MSFT", "TSLA"):
    print(_latest_quote(ticker_symbol))

## Inspect the MCP tools

Run the following command in a terminal from the same directory as the generated server file:

```bash
mcp dev stock_price_server.py
```

The MCP Inspector should list `get_stock_price`, `get_stock_history`, and `compare_stocks`, plus the `stock://{symbol}` resource template. Try `MSFT`, `TSLA`, or another valid ticker.

## Install for Claude Desktop

With Claude Desktop installed, run:

```bash
mcp install stock_price_server.py --name "Stock Analysis Server"
```

Restart Claude Desktop after installation. Local MCP tools can access external services, so only install code you have reviewed and trust.

## Suggested test prompts

1. `What is the latest available closing price for Microsoft?`
2. `Show one month of historical data for TSLA and summarize the direction without giving investment advice.`
3. `Compare the latest closing prices of MSFT and TSLA. State the market date and any data limitations.`
4. `Read the stock://AAPL resource and explain what it contains.`

## Troubleshooting

- **No data returned:** confirm the ticker exists and your environment can reach Yahoo Finance.
- **Tools are missing in Claude Desktop:** restart the app and inspect its extension/developer settings.
- **Import error for `MCPServer`:** ensure MCP Python SDK v2 is installed; v1 used `FastMCP` instead.
- **Inspector cannot start:** verify the `mcp[cli]` extra is installed in the active Python environment.
- **Different currencies:** the comparison tool intentionally avoids a direct percentage comparison without currency conversion.

## Primary references

- [MCP Python SDK v2 documentation](https://py.sdk.modelcontextprotocol.io/)
- [MCP Python SDK v1 maintenance documentation](https://py.sdk.modelcontextprotocol.io/v1/)
- [Official MCP Python SDK repository](https://github.com/modelcontextprotocol/python-sdk)
- [yfinance Ticker.history API](https://ranaroussi.github.io/yfinance/reference/api/yfinance.Ticker.history.html)
- [Claude Desktop local MCP server guidance](https://support.anthropic.com/en/articles/10949351-getting-started-with-local-mcp-servers-on-claude-desktop)